

**Please make sure you use "human_env" kernel**



---

### Manipulate VCF file with BCFtools and VCFtools

You can install BCFtools with conda

`conda install bcftools`

You can install VCFtools with conda

`conda install vcftools`


**Check what we get from last session**

View only header

In [1]:
%cd /Users/noel/Projects/BIO392-Exercises/day_08/sample_A/

import os
os.environ["JAVA_HOME"] = "/Users/noel/miniconda3/envs/bio392/lib/jvm"
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + ":" + os.environ["PATH"]

!bcftools view -h ./output/haplotypecaller/A.g.vcf

/Users/noel/Projects/BIO392-Exercises/day_08/sample_A
##fileformat=VCFv4.2
##FILTER=<ID=PASS,Description="All filters passed">
##ALT=<ID=NON_REF,Description="Represents any possible alternative allele not already represented at this location by REF and ALT">
##FILTER=<ID=LowQual,Description="Low quality">
##FORMAT=<ID=AD,Number=R,Type=Integer,Description="Allelic depths for the ref and alt alleles in the order listed">
##FORMAT=<ID=DP,Number=1,Type=Integer,Description="Approximate read depth (reads with MQ=255 or with bad mates are filtered)">
##FORMAT=<ID=GQ,Number=1,Type=Integer,Description="Genotype Quality">
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
##FORMAT=<ID=MIN_DP,Number=1,Type=Integer,Description="Minimum DP observed within the GVCF block">
##FORMAT=<ID=PGT,Number=1,Type=String,Description="Physical phasing haplotype information, describing how the alternate alleles are phased in relation to one another; will always be heterozygous and is not intended to de

View only data

In [2]:
!bcftools view -H ./output/haplotypecaller/A.g.vcf | head -10

chr14	1	.	N	<NON_REF>	.	.	END=18601020	GT:DP:GQ:MIN_DP:PL	0/0:0:0:0:0,0,0
chr14	18601021	.	C	<NON_REF>	.	.	END=18601022	GT:DP:GQ:MIN_DP:PL	0/0:1:3:1:0,3,37
chr14	18601023	.	T	<NON_REF>	.	.	END=18601027	GT:DP:GQ:MIN_DP:PL	0/0:2:6:2:0,6,74
chr14	18601028	.	A	<NON_REF>	.	.	END=18601034	GT:DP:GQ:MIN_DP:PL	0/0:3:9:3:0,9,107
chr14	18601035	.	T	<NON_REF>	.	.	END=18601039	GT:DP:GQ:MIN_DP:PL	0/0:4:12:4:0,12,154
chr14	18601040	.	A	<NON_REF>	.	.	END=18601045	GT:DP:GQ:MIN_DP:PL	0/0:7:21:7:0,21,257
chr14	18601046	.	G	<NON_REF>	.	.	END=18601046	GT:DP:GQ:MIN_DP:PL	0/0:9:27:9:0,27,350
chr14	18601047	.	A	<NON_REF>	.	.	END=18601050	GT:DP:GQ:MIN_DP:PL	0/0:10:30:10:0,30,367
chr14	18601051	.	C	<NON_REF>	.	.	END=18601052	GT:DP:GQ:MIN_DP:PL	0/0:11:33:11:0,33,408
chr14	18601053	.	T	<NON_REF>	.	.	END=18601056	GT:DP:GQ:MIN_DP:PL	0/0:12:36:12:0,36,440
[main_vcfview] Error: cannot write to (null)


---
# GenomicDBImport

Combine multiple GVCF file



In [3]:
!gatk GenomicsDBImport -h

Using GATK jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar GenomicsDBImport -h
USAGE: GenomicsDBImport [arguments]

Import VCFs to GenomicsDB
Version:4.6.2.0


Required Arguments:

--genomicsdb-update-workspace-path <String>
                              Workspace when updating GenomicsDB. Can be a POSIX file system absolute or relative path
                              or a HDFS/GCS URL. Use this argument when adding new samples to an existing GenomicsDB
                              workspace or when using the output-interval-list-to-file option. Either this or
                              genomicsdb-workspace-path must be specified. Must point to an existing workspace. 
      

---
### Prepare sample name mapping file

It is a tab delimited file with 2 column

sampleID|path/to/sample.g.vcf

---

Use a combination of `ls` `xargs` `basename` `awk` to generate list file contain sample name.

`ls` will list all specifiy target

In [4]:
!ls ./output/haplotypecaller/*.g.vcf

./output/haplotypecaller/A.g.vcf


---
A pipe character `|` has been use to conect two command

`xargs` will read standard input delimited by blanks or new line and executes the command.

`-n1` was use to limit 1 argument per command line

`basename`  take phrase path as input then returns the component
       following the final '/'.

In [5]:
!ls ./output/haplotypecaller/*.g.vcf | xargs -n1 basename

A.g.vcf


Use `awk` a text processing command to split file name.

Chain all commands with pipe `|` and save the result in text file.

In [6]:
!ls ./output/haplotypecaller/*.g.vcf | xargs -n1 basename \
    | awk '{split($0,a,".");print a[1]}' > ./output/haplotypecaller/input_name.txt

In [7]:
!cat ./output/haplotypecaller/input_name.txt

A


---
Use `realpath` command to create list file contain full path of target file

In [8]:
!realpath ./output/haplotypecaller/*.g.vcf > ./output/haplotypecaller/input_path.txt

In [9]:
!cat ./output/haplotypecaller/input_path.txt

/Users/noel/Projects/BIO392-Exercises/day_08/sample_A/output/haplotypecaller/A.g.vcf


Use `paste` command for join files horizontally.
Use `-d` option to set delimeter

In [10]:
!paste ./output/haplotypecaller/input_name.txt \
    ./output/haplotypecaller/input_path.txt  > ./output/haplotypecaller/samples_map.txt

In [11]:
!cat ./output/haplotypecaller/samples_map.txt

A	/Users/noel/Projects/BIO392-Exercises/day_08/sample_A/output/haplotypecaller/A.g.vcf


---
**Create gennomeDB**

In [13]:
!gatk --java-options "-Xms8G -Xmx16G" GenomicsDBImport \
    --genomicsdb-workspace-path ./output/genomeDB \
    --batch-size 50 \
    -L chr14 \
    --sample-name-map ./output/haplotypecaller/samples_map.txt \
    --tmp-dir ./tmp \
    --reader-threads 2

Using GATK jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xms8G -Xmx16G -jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar GenomicsDBImport --genomicsdb-workspace-path ./output/genomeDB --batch-size 50 -L chr14 --sample-name-map ./output/haplotypecaller/samples_map.txt --tmp-dir ./tmp --reader-threads 2
08:01:59.504 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.dylib
08:01:59.516 WARN  NativeLibraryLoader - Unable to load libgkl_compression.dylib from native/libgkl_compression.dylib (/Users/noel/Projects/BIO392-Exercises/day_08/sample_A/tmp/libgkl_compression1458805092419325

---
**(Additional) Update existing genomeDB**

You can add new sample to existing genomeDB by using the same command but with different option.

`--genomicsdb-update-workspace-path`

In [ ]:
"""
!gatk --java-options "-Xms8G -Xmx16G" GenomicsDBImport \
    --genomicsdb-update-workspace-path ./output/genomeDB \
    --batch-size 50 \
    -L chr14 \
    --sample-name-map ./ouptut/haplotypecaller/new_samples_map.txt \
    --tmp-dir ./tmp \
    --reader-threads 2
"""

---
# GenotypeGVCFS

Perform joint genotyping on single or multiple sample pre-called with HaplotypeCaller

**Single sample**

In [14]:
!gatk GenotypeGVCFs -h

Using GATK jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar GenotypeGVCFs -h
USAGE: GenotypeGVCFs [arguments]

Perform joint genotyping on a single-sample GVCF from HaplotypeCaller or a multi-sample GVCF from CombineGVCFs or
GenomicsDBImport
Version:4.6.2.0


Required Arguments:

--output,-O <GATKPath>        File to which variants should be written  Required. 

--reference,-R <GATKPath>     Reference sequence file  Required. 

--variant,-V <String>         A VCF file containing variants  Required. 


Optional Arguments:

--add-output-sam-program-record <Boolean>
                              If true, adds a PG tag to created SAM/BAM/CRAM files.  Default value: true. Possible
      

In [15]:
!mkdir -p ./output/genotypegvcf

In [16]:
!gatk GenotypeGVCFs -h

Using GATK jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar GenotypeGVCFs -h
USAGE: GenotypeGVCFs [arguments]

Perform joint genotyping on a single-sample GVCF from HaplotypeCaller or a multi-sample GVCF from CombineGVCFs or
GenomicsDBImport
Version:4.6.2.0


Required Arguments:

--output,-O <GATKPath>        File to which variants should be written  Required. 

--reference,-R <GATKPath>     Reference sequence file  Required. 

--variant,-V <String>         A VCF file containing variants  Required. 


Optional Arguments:

--add-output-sam-program-record <Boolean>
                              If true, adds a PG tag to created SAM/BAM/CRAM files.  Default value: true. Possible
      

In [17]:
!gatk --java-options "-Xmx8G" GenotypeGVCFs \
    -R ./reference/chr14.fa \
    -V ./output/haplotypecaller/A.g.vcf \
    -O ./output/genotypegvcf/A.vcf \
    --tmp-dir ./tmp

Using GATK jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar GenotypeGVCFs -R ./reference/chr14.fa -V ./output/haplotypecaller/A.g.vcf -O ./output/genotypegvcf/A.vcf --tmp-dir ./tmp
08:02:31.341 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.dylib
08:02:31.353 WARN  NativeLibraryLoader - Unable to load libgkl_compression.dylib from native/libgkl_compression.dylib (/Users/noel/Projects/BIO392-Exercises/day_08/sample_A/tmp/libgkl_compression2548092562185106330.dylib: dlopen(/Users/noel/Projects/BIO392-Exercises/day_08/sample

In [18]:
!bcftools view -H ./output/genotypegvcf/A.vcf | head -10

chr14	18601430	.	A	C	857.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-6.99;DP=119;ExcessHet=0;FS=99.112;MLEAC=1;MLEAF=0.5;MQ=33.35;MQRankSum=-0.422;QD=7.59;ReadPosRankSum=-1.211;SOR=7.167	GT:AD:DP:GQ:PL	0/1:78,35:113:99:865,0,2376
chr14	18601553	.	T	A	4224.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-5.011;DP=356;ExcessHet=0;FS=54.216;MLEAC=1;MLEAF=0.5;MQ=37.6;MQRankSum=-4.832;QD=13.16;ReadPosRankSum=-2.075;SOR=2.467	GT:AD:DP:GQ:PL	0/1:167,154:321:99:4232,0,4662
chr14	18601712	.	G	T	1614.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-3.946;DP=122;ExcessHet=0;FS=15.631;MLEAC=1;MLEAF=0.5;MQ=44.86;MQRankSum=-9.441;QD=13.8;ReadPosRankSum=0.06;SOR=4.086	GT:AD:DP:GQ:PL	0/1:59,58:117:99:1622,0,1919
chr14	18601819	.	C	A	425.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-4.483;DP=36;ExcessHet=0;FS=48.378;MLEAC=1;MLEAF=0.5;MQ=32.98;MQRankSum=4.23;QD=12.9;ReadPosRankSum=0.036;SOR=5.417	GT:AD:DP:GQ:PL	0/1:19,14:33:99:433,0,524
chr14	18601844	.	T	C	184.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-2.245;DP=69;ExcessHet=0;FS=51.431;MLEAC=1;MLEAF=0.

**Multiple sample**

In [19]:
!gatk --java-options "-Xmx8G" GenotypeGVCFs \
    -R ./reference/chr14.fa \
    -V gendb://${PWD}/output/genomeDB \
    -O ./output/genotypegvcf/combine.vcf \
    --tmp-dir ./tmp

Using GATK jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar GenotypeGVCFs -R ./reference/chr14.fa -V gendb:///Users/noel/Projects/BIO392-Exercises/day_08/sample_A/output/genomeDB -O ./output/genotypegvcf/combine.vcf --tmp-dir ./tmp
08:02:46.875 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.dylib
08:02:46.886 WARN  NativeLibraryLoader - Unable to load libgkl_compression.dylib from native/libgkl_compression.dylib (/Users/noel/Projects/BIO392-Exercises/day_08/sample_A/tmp/libgkl_compression9103930020850024582.dylib: dlopen(


**View VCF file**

In [20]:
!bcftools view -h ./output/genotypegvcf/combine.vcf | tail

##INFO=<ID=RAW_MQandDP,Number=2,Type=Integer,Description="Raw data (sum of squared MQ and total depth) for improved RMS Mapping Quality calculation. Incompatible with deprecated RAW_MQ formulation.">
##INFO=<ID=ReadPosRankSum,Number=1,Type=Float,Description="Z-score from Wilcoxon rank sum test of Alt vs. Ref read position bias">
##INFO=<ID=SOR,Number=1,Type=Float,Description="Symmetric Odds Ratio of 2x2 contingency table to detect strand bias">
##contig=<ID=chr14,length=107043718>
##source=GenomicsDBImport
##source=GenotypeGVCFs
##source=HaplotypeCaller
##bcftools_viewVersion=1.23.1+htslib-1.23.1
##bcftools_viewCommand=view -h ./output/genotypegvcf/combine.vcf; Date=Tue May  5 08:03:03 2026
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO	FORMAT	A


In [21]:
!bcftools view -H ./output/genotypegvcf/combine.vcf | head -10

chr14	18601430	.	A	C	857.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-6.99;DP=119;ExcessHet=0;FS=99.112;MLEAC=1;MLEAF=0.5;MQ=33.35;MQRankSum=-0.422;QD=7.59;ReadPosRankSum=-1.211;SOR=7.167	GT:AD:DP:GQ:PL	0/1:78,35:113:99:865,0,2376
chr14	18601553	.	T	A	4224.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-5.011;DP=356;ExcessHet=0;FS=54.216;MLEAC=1;MLEAF=0.5;MQ=37.6;MQRankSum=-4.832;QD=13.16;ReadPosRankSum=-2.075;SOR=2.467	GT:AD:DP:GQ:PL	0/1:167,154:321:99:4232,0,4662
chr14	18601712	.	G	T	1614.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-3.946;DP=122;ExcessHet=0;FS=15.631;MLEAC=1;MLEAF=0.5;MQ=44.86;MQRankSum=-9.441;QD=13.8;ReadPosRankSum=0.06;SOR=4.086	GT:AD:DP:GQ:PL	0/1:59,58:117:99:1622,0,1919
chr14	18601819	.	C	A	425.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-4.483;DP=36;ExcessHet=0;FS=48.378;MLEAC=1;MLEAF=0.5;MQ=32.98;MQRankSum=4.23;QD=12.9;ReadPosRankSum=0.036;SOR=5.417	GT:AD:DP:GQ:PL	0/1:19,14:33:99:433,0,524
chr14	18601844	.	T	C	184.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-2.245;DP=69;ExcessHet=0;FS=51.431;MLEAC=1;MLEAF=0.

---
# Variant Quality Score Recalibration (VQSR)

VQSR have 2 step.

    1. Create model with command `VariantRecalibrator`. Run separately for SNP and INDEL
    
    2. Apply model to data with command 'ApplyVQSR'

In [22]:
!gatk VariantRecalibrator -h

Using GATK jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar VariantRecalibrator -h
USAGE: VariantRecalibrator [arguments]

Build a recalibration model to score variant quality for filtering purposes
Version:4.6.2.0


Required Arguments:

--output,-O <GATKPath>        The output recal file used by ApplyVQSR  Required. 

--resource <FeatureInput>     A list of sites for which to apply a prior probability of being correct but which aren't
                              used by the algorithm (training and truth sets are required to run)  This argument must be
                              specified at least once. Required. 

--tranches-file <File>        The output tranches file used by 

In [23]:
!gatk ApplyVQSR -h

Using GATK jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar ApplyVQSR -h
USAGE: ApplyVQSR [arguments]

Apply a score cutoff to filter variants based on a recalibration table
Version:4.6.2.0


Required Arguments:

--output,-O <GATKPath>        The output filtered and recalibrated VCF file in which each variant is annotated with its
                              VQSLOD value  Required. 

--recal-file <FeatureInput>   The input recal file used by ApplyVQSR  Required. 

--variant,-V <GATKPath>       One or more VCF files containing variants  This argument must be specified at least once.
                              Required. 


Optional Arguments:

--add-output-sam-program-record <Boo

Create output directory

In [24]:
!mkdir -p ./output/vqsr

## Create model for SNP

Based on recommendation from GATK website

https://gatk.broadinstitute.org/hc/en-us/articles/4402736812443-Which-training-sets-arguments-should-I-use-for-running-VQSR

The standard set are store in

https://console.cloud.google.com/storage/browser/gcp-public-data--broad-references/hg38/v0?pageState=(%22StorageObjectListTable%22:(%22f%22:%22%255B%255D%22))

In [25]:
!gatk --java-options "-Xmx8G" VariantRecalibrator \
    -R ./reference/chr14.fa \
    -V ./output/genotypegvcf/combine.vcf \
    --resource:hapmap,known=false,training=true,truth=true,prior=15.0 ./db/vqsr/resources_broad_hg38_v0_hapmap_3.3.hg38.vcf.gz  \
    --resource:omni,known=false,training=true,truth=false,prior=12.0 ./db/vqsr/resources_broad_hg38_v0_1000G_omni2.5.hg38.vcf.gz \
    --resource:1000G,known=false,training=true,truth=false,prior=10.0 ./db/vqsr/resources_broad_hg38_v0_1000G_phase1.snps.high_confidence.hg38_chr14roi.vcf.gz \
    --resource:dbsnp,known=true,training=false,truth=false,prior=2.0 ./db/Homo_sapiens_assembly38.dbsnp138_roi.vcf.gz \
    -an QD -an MQ -an MQRankSum -an ReadPosRankSum -an FS -an SOR  \
    -mode SNP \
    --max-gaussians 4 \
    -O ./output/vqsr/vqsr_snp.recal \
    --tranches-file ./output/vqsr/vqsr_snp.tranches

Using GATK jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar VariantRecalibrator -R ./reference/chr14.fa -V ./output/genotypegvcf/combine.vcf --resource:hapmap,known=false,training=true,truth=true,prior=15.0 ./db/vqsr/resources_broad_hg38_v0_hapmap_3.3.hg38.vcf.gz --resource:omni,known=false,training=true,truth=false,prior=12.0 ./db/vqsr/resources_broad_hg38_v0_1000G_omni2.5.hg38.vcf.gz --resource:1000G,known=false,training=true,truth=false,prior=10.0 ./db/vqsr/resources_broad_hg38_v0_1000G_phase1.snps.high_confidence.hg38_chr14roi.vcf.gz --resource:dbsnp,known=true,training=false,truth=false,prior=2.0 ./db/Homo_sapiens_assembly38.dbsnp138_roi.vcf.gz -an QD -an MQ -an MQRankSu

## Create model for INDEL

In [26]:
!gatk --java-options "-Xmx8G" VariantRecalibrator \
    -R ./reference/chr14.fa \
    -V ./output/genotypegvcf/combine.vcf \
    --resource:mills,known=false,training=true,truth=true,prior=12.0 ./db/resources_broad_hg38_v0_Mills_and_1000G_gold_standard.indels.hg38.vcf.gz \
    --resource:dbsnp,known=true,training=false,truth=false,prior=2.0 ./db/Homo_sapiens_assembly38.dbsnp138_roi.vcf.gz \
    -an QD -an MQRankSum -an ReadPosRankSum -an FS -an SOR -an DP \
    -mode INDEL \
    --max-gaussians 1 \
    -O ./output/vqsr/vqsr_indel.recal \
    --tranches-file ./output/vqsr/vqsr_indel.tranches

Using GATK jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar VariantRecalibrator -R ./reference/chr14.fa -V ./output/genotypegvcf/combine.vcf --resource:mills,known=false,training=true,truth=true,prior=12.0 ./db/resources_broad_hg38_v0_Mills_and_1000G_gold_standard.indels.hg38.vcf.gz --resource:dbsnp,known=true,training=false,truth=false,prior=2.0 ./db/Homo_sapiens_assembly38.dbsnp138_roi.vcf.gz -an QD -an MQRankSum -an ReadPosRankSum -an FS -an SOR -an DP -mode INDEL --max-gaussians 1 -O ./output/vqsr/vqsr_indel.recal --tranches-file ./output/vqsr/vqsr_indel.tranches
08:04:45.634 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/Users/noel/miniconda3

## Apply VQSR SNP model

In [27]:
!gatk --java-options "-Xmx8G" ApplyVQSR \
  -V ./output/genotypegvcf/combine.vcf \
  --recal-file ./output/vqsr/vqsr_snp.recal \
  -mode SNP \
  --tranches-file ./output/vqsr/vqsr_snp.tranches \
  --truth-sensitivity-filter-level 99.5 \
  --create-output-variant-index true \
  -O ./output/vqsr/combine_snp_recalibrated.vcf.gz

Using GATK jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar ApplyVQSR -V ./output/genotypegvcf/combine.vcf --recal-file ./output/vqsr/vqsr_snp.recal -mode SNP --tranches-file ./output/vqsr/vqsr_snp.tranches --truth-sensitivity-filter-level 99.5 --create-output-variant-index true -O ./output/vqsr/combine_snp_recalibrated.vcf.gz
08:04:51.366 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.dylib
08:04:51.376 WARN  NativeLibraryLoader - Unable to load libgkl_compression.dylib from native/libgkl_compression.dylib (/private/var/

## Apply VQSR INDEL model

In [28]:
!gatk --java-options "-Xmx8G" ApplyVQSR \
  -V ./output/vqsr/combine_snp_recalibrated.vcf.gz \
  -mode INDEL \
  --recal-file ./output/vqsr/vqsr_indel.recal \
  --tranches-file ./output/vqsr/vqsr_indel.tranches \
  --truth-sensitivity-filter-level 99.0 \
  --create-output-variant-index true \
  -O ./output/vqsr/combine_indel_snp_recalibrated.vcf.gz

Using GATK jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar ApplyVQSR -V ./output/vqsr/combine_snp_recalibrated.vcf.gz -mode INDEL --recal-file ./output/vqsr/vqsr_indel.recal --tranches-file ./output/vqsr/vqsr_indel.tranches --truth-sensitivity-filter-level 99.0 --create-output-variant-index true -O ./output/vqsr/combine_indel_snp_recalibrated.vcf.gz
08:04:56.137 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/Users/noel/miniconda3/envs/bio392/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.dylib
08:04:56.149 WARN  NativeLibraryLoader - Unable to load libgkl_compression.dylib from native/libgkl_compress

### Gatk Hard filtering (Alternative)

Instead of using VQSR. GATK also provide a hard filtering setting.

https://gatk.broadinstitute.org/hc/en-us/articles/360035890471-Hard-filtering-germline-short-variants

In [ ]:
"""
!gatk VariantFiltration \
     -V ./output/genotypegvcf/combine.vcf \
     -O ./output/genotypegvcf/combine_filtered_variants.vcf.gz \
     -filter "QD < 2.0" --filter-name "QD2" \
     -filter "MQ < 40.0" --filter-name "MQ40" \
     -filter "FS > 60.0" --filter-name "FS60" \
     -filter "SOR > 3.0" --filter-name "SOR3" \
     -filter "MQRankSum < -12.5" --filter-name "MQRankSum-12.5" \
     -filter "ReadPosRankSum < -8.0" --filter-name "ReadPosRankSum-8"
"""

## View your VCF file

In [29]:
!bcftools view -H ./output/vqsr/combine_indel_snp_recalibrated.vcf.gz | head -10

chr14	18601430	.	A	C	857.64	VQSRTrancheSNP99.90to100.00	AC=1;AF=0.5;AN=2;BaseQRankSum=-6.99;DP=119;ExcessHet=0;FS=99.112;MLEAC=1;MLEAF=0.5;MQ=33.35;MQRankSum=-0.422;QD=7.59;ReadPosRankSum=-1.211;SOR=7.167;VQSLOD=-117.2;culprit=FS	GT:AD:DP:GQ:PL	0/1:78,35:113:99:865,0,2376
chr14	18601553	.	T	A	4224.64	VQSRTrancheSNP99.90to100.00	AC=1;AF=0.5;AN=2;BaseQRankSum=-5.011;DP=356;ExcessHet=0;FS=54.216;MLEAC=1;MLEAF=0.5;MQ=37.6;MQRankSum=-4.832;QD=13.16;ReadPosRankSum=-2.075;SOR=2.467;VQSLOD=-42.05;culprit=FS	GT:AD:DP:GQ:PL	0/1:167,154:321:99:4232,0,4662
chr14	18601712	.	G	T	1614.64	VQSRTrancheSNP99.90to100.00	AC=1;AF=0.5;AN=2;BaseQRankSum=-3.946;DP=122;ExcessHet=0;FS=15.631;MLEAC=1;MLEAF=0.5;MQ=44.86;MQRankSum=-9.441;QD=13.8;ReadPosRankSum=0.06;SOR=4.086;VQSLOD=-16.68;culprit=MQRankSum	GT:AD:DP:GQ:PL	0/1:59,58:117:99:1622,0,1919
chr14	18601819	.	C	A	425.64	VQSRTrancheSNP99.90to100.00	AC=1;AF=0.5;AN=2;BaseQRankSum=-4.483;DP=36;ExcessHet=0;FS=48.378;MLEAC=1;MLEAF=0.5;MQ=32.98;MQRankSum=4.23;QD=12

---

### Filter VCF with vcftools

In [30]:
!vcftools --gzvcf ./output/vqsr/combine_indel_snp_recalibrated.vcf.gz \
    --recode \
    --recode-INFO-all \
    --remove-filtered-all \
    --out ./output/vqsr/combine_indel_snp_recalibrated


VCFtools - 0.1.17
(C) Adam Auton and Anthony Marcketta 2009

Parameters as interpreted:
	--gzvcf ./output/vqsr/combine_indel_snp_recalibrated.vcf.gz
	--recode-INFO-all
	--out ./output/vqsr/combine_indel_snp_recalibrated
	--recode
	--remove-filtered-all

Using zlib version: 1.3.1
After filtering, kept 1 out of 1 Individuals
Outputting VCF file...
After filtering, kept 2671 out of a possible 2799 Sites
Run Time = 0.00 seconds


**BCFtools equivalent command**

In [ ]:
"""
!bcftools view \
    -m 2 -M 2 \
    -f PASS \
    -O v \
    -o ./output/vqsr/combine_indel_snp_recalibrated.vcf \
    ./output/vqsr/combine_indel_snp_recalibrated.vcf.gz
""""

### Check VCF file before and after filtering

In [31]:
!bcftools view -H ./output/vqsr/combine_indel_snp_recalibrated.vcf.gz | head -10

chr14	18601430	.	A	C	857.64	VQSRTrancheSNP99.90to100.00	AC=1;AF=0.5;AN=2;BaseQRankSum=-6.99;DP=119;ExcessHet=0;FS=99.112;MLEAC=1;MLEAF=0.5;MQ=33.35;MQRankSum=-0.422;QD=7.59;ReadPosRankSum=-1.211;SOR=7.167;VQSLOD=-117.2;culprit=FS	GT:AD:DP:GQ:PL	0/1:78,35:113:99:865,0,2376
chr14	18601553	.	T	A	4224.64	VQSRTrancheSNP99.90to100.00	AC=1;AF=0.5;AN=2;BaseQRankSum=-5.011;DP=356;ExcessHet=0;FS=54.216;MLEAC=1;MLEAF=0.5;MQ=37.6;MQRankSum=-4.832;QD=13.16;ReadPosRankSum=-2.075;SOR=2.467;VQSLOD=-42.05;culprit=FS	GT:AD:DP:GQ:PL	0/1:167,154:321:99:4232,0,4662
chr14	18601712	.	G	T	1614.64	VQSRTrancheSNP99.90to100.00	AC=1;AF=0.5;AN=2;BaseQRankSum=-3.946;DP=122;ExcessHet=0;FS=15.631;MLEAC=1;MLEAF=0.5;MQ=44.86;MQRankSum=-9.441;QD=13.8;ReadPosRankSum=0.06;SOR=4.086;VQSLOD=-16.68;culprit=MQRankSum	GT:AD:DP:GQ:PL	0/1:59,58:117:99:1622,0,1919
chr14	18601819	.	C	A	425.64	VQSRTrancheSNP99.90to100.00	AC=1;AF=0.5;AN=2;BaseQRankSum=-4.483;DP=36;ExcessHet=0;FS=48.378;MLEAC=1;MLEAF=0.5;MQ=32.98;MQRankSum=4.23;QD=12

In [32]:
!bcftools view -H ./output/vqsr/combine_indel_snp_recalibrated.recode.vcf | head -10

chr14	18985420	.	T	C	54.64	PASS	AC=1;AF=0.5;AN=2;BaseQRankSum=0;DP=3;ExcessHet=0;FS=0;MLEAC=1;MLEAF=0.5;MQ=36.19;MQRankSum=-0.431;QD=18.21;ReadPosRankSum=0.967;SOR=1.179;VQSLOD=-0.0063;culprit=MQ	GT:AD:DP:GQ:PL	0/1:1,2:3:30:62,0,30
chr14	19180623	.	AT	A	286	PASS	AC=2;AF=1;AN=2;DP=8;ExcessHet=0;FS=0;MLEAC=2;MLEAF=1;MQ=42;QD=25.36;SOR=4.407;VQSLOD=7.47;culprit=QD	GT:AD:DP:GQ:PL	1/1:0,8:8:24:300,24,0
chr14	19180816	.	T	TG	212.93	PASS	AC=2;AF=1;AN=2;DP=6;ExcessHet=0;FS=0;MLEAC=2;MLEAF=1;MQ=34.87;QD=28.73;SOR=1.329;VQSLOD=2.25;culprit=SOR	GT:AD:DP:GQ:PL	1/1:0,6:6:18:227,18,0
chr14	19413462	.	G	C	102.64	PASS	AC=1;AF=0.5;AN=2;BaseQRankSum=-3.401;DP=42;ExcessHet=0;FS=0;MLEAC=1;MLEAF=0.5;MQ=26.1;MQRankSum=0.742;QD=2.77;ReadPosRankSum=-1.689;SOR=0.064;VQSLOD=1.3;culprit=MQ	GT:AD:DP:GQ:PL	0/1:30,7:37:99:110,0,829
chr14	19433743	.	C	T	319.64	PASS	AC=1;AF=0.5;AN=2;BaseQRankSum=-3.178;DP=42;ExcessHet=0;FS=0;MLEAC=1;MLEAF=0.5;MQ=26.87;MQRankSum=0.802;QD=8.2;ReadPosRankSum=-2.124;SOR=0.287;VQSLOD=2.13

---
**Compress and index VCF file**

In [33]:
!bgzip -c ./output/vqsr/combine_indel_snp_recalibrated.recode.vcf > ./output/vqsr/combine_indel_snp_recalibrated.recode.vcf.gz

In [34]:
!tabix -p vcf ./output/vqsr/combine_indel_snp_recalibrated.recode.vcf.gz

---

## Exercise : Run the whole pipeline on sampleB

- Sequence QC
- Alignemnt
- Alignemnt post process
- Variant calling
- Create combine VCF between sampleA and sampleB
- Variant filtering

---
